
# Roxy notebook example: Residue distribution descriptors along the sequence

This notebook is a **reference implementation example** for the **residue distribution descriptor family** in Roxy.

These descriptors summarize **how residues or residue groups are distributed across different regions of the sequence**, rather than only asking where the first or last occurrence appears.

## Covered outputs

This notebook implements examples such as:

- fraction of residues in N / middle / C segments
- quartile-based and tercile-based distribution profiles
- residue-group occupancy by bins
- cumulative distribution summaries
- regional enrichment scores
- segment entropy across bins
- concentration of residues in the most occupied bin
- distribution spread across bins
- class-style implementation for later migration into Roxy

These descriptors are useful because two sequences can have similar composition and similar mean position, but very different **regional distributions**.


In [1]:

from collections import Counter

import numpy as np
import pandas as pd


## Demo dataset

In [2]:

df_demo = pd.DataFrame(
    {
        "sequence_id": [
            "dist_1",
            "dist_2",
            "dist_3",
            "dist_4",
            "dist_5",
            "dist_6",
        ],
        "sequence": [
            "MKWVTFISLLFLFSSAYSRGVFRR",
            "GGGGGGGGGGGGGGG",
            "KRRKRRKRRKRRDDDDEE",
            "ACDEFGHIKLMNPQRSTVWY",
            "PPPPGSSSSSTTTTNNQQQ",
            "MSTNPKPQRITLKDGNKVELV",
        ],
        "label": ["A", "B", "A", "B", "A", "B"],
    }
)

df_demo


,sequence_id,sequence,label
0,dist_1,MKWVTFISLLFLFSSAYSRGVFRR,A
1,dist_2,GGGGGGGGGGGGGGG,B
2,dist_3,KRRKRRKRRKRRDDDDEE,A
3,dist_4,ACDEFGHIKLMNPQRSTVWY,B
4,dist_5,PPPPGSSSSSTTTTNNQQQ,A
5,dist_6,MSTNPKPQRITLKDGNKVELV,B


## Constants

In [3]:

STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

AA_GROUPS = {
    "positive": set("KRH"),
    "negative": set("DE"),
    "charged": set("KRHDE"),
    "polar": set("STNQCYWHKRDE"),
    "nonpolar": set("AVLIMFGP"),
    "aromatic": set("FWYH"),
    "aliphatic": set("AVLIM"),
    "hydrophobic": set("AVLIMFWCY"),
    "hydrophilic": set("RNDQEHKST"),
    "disorder_promoting": set("ARGQSEPK"),
    "order_promoting": set("CWYFILNV"),
}

AA_SINGLETS = {
    "K": set("K"),
    "R": set("R"),
    "D": set("D"),
    "E": set("E"),
    "G": set("G"),
    "P": set("P"),
    "W": set("W"),
    "Y": set("Y"),
}


## Helper functions

In [4]:

def clean_sequence(seq: str) -> str:
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper().replace("*", "")
    return "".join([aa for aa in seq if aa in STANDARD_AA])


def positions_of_group(seq: str, aa_group):
    return [i + 1 for i, aa in enumerate(seq) if aa in aa_group]


def normalized_positions(positions, seq_len: int):
    if seq_len == 0:
        return []
    return [p / seq_len for p in positions]


def bin_edges(n_bins: int):
    return np.linspace(0.0, 1.0, n_bins + 1)


def bin_occupancy(norm_positions, n_bins: int):
    if len(norm_positions) == 0:
        return np.zeros(n_bins, dtype=float)
    edges = bin_edges(n_bins)
    counts = np.zeros(n_bins, dtype=float)

    for pos in norm_positions:
        idx = np.searchsorted(edges, pos, side="right") - 1
        idx = min(max(idx, 0), n_bins - 1)
        counts[idx] += 1

    counts /= len(norm_positions)
    return counts


def occupancy_entropy(occupancy):
    occupancy = np.array(occupancy, dtype=float)
    if occupancy.sum() == 0:
        return np.nan
    probs = occupancy[occupancy > 0]
    return float(-(probs * np.log2(probs)).sum())


def concentration_max_bin(occupancy):
    occupancy = np.array(occupancy, dtype=float)
    if occupancy.sum() == 0:
        return np.nan
    return float(np.max(occupancy))


def distribution_spread(occupancy):
    occupancy = np.array(occupancy, dtype=float)
    if occupancy.sum() == 0:
        return np.nan
    non_zero_bins = np.sum(occupancy > 0)
    return float(non_zero_bins / len(occupancy))


def regional_enrichment(occupancy, region_index: int):
    occupancy = np.array(occupancy, dtype=float)
    if occupancy.sum() == 0:
        return np.nan
    uniform = 1.0 / len(occupancy)
    return float(occupancy[region_index] - uniform)


def cumulative_occupancy(occupancy):
    occupancy = np.array(occupancy, dtype=float)
    return np.cumsum(occupancy)


def summarize_distribution(norm_positions, seq_len: int, prefix: str, n_bins: int):
    occupancy = bin_occupancy(norm_positions, n_bins=n_bins)
    cumulative = cumulative_occupancy(occupancy)

    out = {}
    for i, value in enumerate(occupancy, start=1):
        out[f"{prefix}_bin{i}_frac"] = float(value)

    for i, value in enumerate(cumulative, start=1):
        out[f"{prefix}_cum{i}"] = float(value)

    out[f"{prefix}_entropy"] = occupancy_entropy(occupancy)
    out[f"{prefix}_max_bin_frac"] = concentration_max_bin(occupancy)
    out[f"{prefix}_spread"] = distribution_spread(occupancy)

    for i in range(n_bins):
        out[f"{prefix}_bin{i+1}_enrichment"] = regional_enrichment(occupancy, i)

    return out


## Core descriptor function

In [5]:

def distribution_descriptors(seq: str) -> dict:
    seq = clean_sequence(seq)
    seq_len = len(seq)

    out = {
        "dist_length": seq_len,
        "dist_valid_residue_count": seq_len,
    }

    if seq_len == 0:
        return out

    tracked_groups = {
        "charged": AA_GROUPS["charged"],
        "hydrophobic": AA_GROUPS["hydrophobic"],
        "aromatic": AA_GROUPS["aromatic"],
        "polar": AA_GROUPS["polar"],
        "positive": AA_GROUPS["positive"],
        "negative": AA_GROUPS["negative"],
        "gly": AA_SINGLETS["G"],
        "pro": AA_SINGLETS["P"],
    }

    for name, group in tracked_groups.items():
        pos = positions_of_group(seq, group)
        norm_pos = normalized_positions(pos, seq_len)

        out[f"dist_{name}_count"] = len(pos)
        out.update(summarize_distribution(norm_pos, seq_len, prefix=f"dist_{name}_tercile", n_bins=3))
        out.update(summarize_distribution(norm_pos, seq_len, prefix=f"dist_{name}_quartile", n_bins=4))

    return out


## Functional usage on one sequence

In [6]:

example = distribution_descriptors(df_demo.loc[0, "sequence"])
list(example.items())[:20]


[('dist_length', 24),
 ('dist_valid_residue_count', 24),
 ('dist_charged_count', 4),
 ('dist_charged_tercile_bin1_frac', 0.25),
 ('dist_charged_tercile_bin2_frac', 0.0),
 ('dist_charged_tercile_bin3_frac', 0.75),
 ('dist_charged_tercile_cum1', 0.25),
 ('dist_charged_tercile_cum2', 0.25),
 ('dist_charged_tercile_cum3', 1.0),
 ('dist_charged_tercile_entropy', 0.8112781244591328),
 ('dist_charged_tercile_max_bin_frac', 0.75),
 ('dist_charged_tercile_spread', 0.6666666666666666),
 ('dist_charged_tercile_bin1_enrichment', -0.08333333333333331),
 ('dist_charged_tercile_bin2_enrichment', -0.3333333333333333),
 ('dist_charged_tercile_bin3_enrichment', 0.4166666666666667),
 ('dist_charged_quartile_bin1_frac', 0.25),
 ('dist_charged_quartile_bin2_frac', 0.0),
 ('dist_charged_quartile_bin3_frac', 0.0),
 ('dist_charged_quartile_bin4_frac', 0.75),
 ('dist_charged_quartile_cum1', 0.25)]

## Apply distribution descriptors to the full dataset

In [7]:

df_dist = pd.concat(
    [
        df_demo,
        df_demo["sequence"].apply(distribution_descriptors).apply(pd.Series),
    ],
    axis=1,
)

df_dist.head()


,sequence_id,sequence,label,dist_length,dist_valid_residue_count,dist_charged_count,dist_charged_tercile_bin1_frac,dist_charged_tercile_bin2_frac,dist_charged_tercile_bin3_frac,dist_charged_tercile_cum1,...,dist_pro_quartile_cum2,dist_pro_quartile_cum3,dist_pro_quartile_cum4,dist_pro_quartile_entropy,dist_pro_quartile_max_bin_frac,dist_pro_quartile_spread,dist_pro_quartile_bin1_enrichment,dist_pro_quartile_bin2_enrichment,dist_pro_quartile_bin3_enrichment,dist_pro_quartile_bin4_enrichment
0,dist_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24.0,24.0,4.0,0.250000,0.000000,0.750000,0.250000,...,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,dist_2,GGGGGGGGGGGGGGG,B,15.0,15.0,0.0,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,dist_3,KRRKRRKRRKRRDDDDEE,A,18.0,18.0,18.0,0.277778,0.333333,0.388889,0.277778,...,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,dist_4,ACDEFGHIKLMNPQRSTVWY,B,20.0,20.0,5.0,0.400000,0.400000,0.200000,0.400000,...,0.0,1.0,1.0,-0.0,1.0,0.25,-0.25,-0.25,0.75,-0.25
4,dist_5,PPPPGSSSSSTTTTNNQQQ,A,19.0,19.0,0.0,0.000000,0.000000,0.000000,0.000000,...,1.0,1.0,1.0,-0.0,1.0,0.25,0.75,-0.25,-0.25,-0.25


## Inspect distribution descriptor columns

In [8]:

dist_cols = [c for c in df_dist.columns if c.startswith("dist_") and c not in {"dist_length", "dist_valid_residue_count"}]
len(dist_cols), dist_cols[:20]


(224,
 ['dist_charged_count',
  'dist_charged_tercile_bin1_frac',
  'dist_charged_tercile_bin2_frac',
  'dist_charged_tercile_bin3_frac',
  'dist_charged_tercile_cum1',
  'dist_charged_tercile_cum2',
  'dist_charged_tercile_cum3',
  'dist_charged_tercile_entropy',
  'dist_charged_tercile_max_bin_frac',
  'dist_charged_tercile_spread',
  'dist_charged_tercile_bin1_enrichment',
  'dist_charged_tercile_bin2_enrichment',
  'dist_charged_tercile_bin3_enrichment',
  'dist_charged_quartile_bin1_frac',
  'dist_charged_quartile_bin2_frac',
  'dist_charged_quartile_bin3_frac',
  'dist_charged_quartile_bin4_frac',
  'dist_charged_quartile_cum1',
  'dist_charged_quartile_cum2',
  'dist_charged_quartile_cum3'])

In [9]:

df_dist[
    [
        "sequence_id",
        "dist_charged_tercile_bin1_frac",
        "dist_charged_tercile_bin2_frac",
        "dist_charged_tercile_bin3_frac",
        "dist_hydrophobic_quartile_bin1_frac",
        "dist_hydrophobic_quartile_bin4_frac",
        "dist_aromatic_quartile_entropy",
        "dist_gly_tercile_max_bin_frac",
    ]
]


,sequence_id,dist_charged_tercile_bin1_frac,dist_charged_tercile_bin2_frac,dist_charged_tercile_bin3_frac,dist_hydrophobic_quartile_bin1_frac,dist_hydrophobic_quartile_bin4_frac,dist_aromatic_quartile_entropy,dist_gly_tercile_max_bin_frac
0,dist_1,0.250000,0.000000,0.750000,0.214286,0.142857,1.918296,1.0
1,dist_2,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.4
2,dist_3,0.277778,0.333333,0.388889,0.000000,0.000000,NaN,NaN
3,dist_4,0.400000,0.400000,0.200000,0.222222,0.333333,1.000000,1.0
4,dist_5,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,1.0
5,dist_6,0.166667,0.333333,0.500000,0.166667,0.500000,NaN,1.0


## Dataset-level summary

In [10]:

dist_summary = (
    df_dist[dist_cols]
    .mean(axis=0, numeric_only=True)
    .sort_values(ascending=False)
    .rename("mean_value")
    .reset_index()
    .rename(columns={"index": "descriptor"})
)

dist_summary.head(15)


,descriptor,mean_value
0,dist_polar_count,11.166667
1,dist_charged_count,5.500000
2,dist_hydrophobic_count,4.833333
3,dist_positive_count,3.833333
4,dist_gly_count,3.166667
5,dist_hydrophobic_quartile_entropy,1.897198
6,dist_polar_quartile_entropy,1.864807
7,dist_aromatic_count,1.666667
8,dist_negative_count,1.666667
9,dist_aromatic_tercile_entropy,1.542481


## Sanity checks

In [11]:

assert "dist_charged_tercile_bin1_frac" in df_dist.columns
assert "dist_hydrophobic_quartile_bin4_frac" in df_dist.columns
assert "dist_aromatic_quartile_entropy" in df_dist.columns
assert "dist_positive_tercile_bin1_enrichment" in df_dist.columns
assert "dist_negative_quartile_cum4" in df_dist.columns
assert df_dist["dist_length"].min() > 0

print(f"Number of distribution descriptor columns: {len(dist_cols)}")
print("Residue distribution descriptor checks passed.")


Number of distribution descriptor columns: 224
Residue distribution descriptor checks passed.


## Class-style implementation closer to the real package

In [12]:

class ResidueDistributionDescriptors:
    """Example class-style residue-distribution implementation for later migration into Roxy."""

    def transform_sequence(self, seq: str) -> dict:
        return distribution_descriptors(seq)

    def transform(self, sequences) -> pd.DataFrame:
        return pd.DataFrame([self.transform_sequence(seq) for seq in sequences])


dist_transformer = ResidueDistributionDescriptors()
dist_matrix = dist_transformer.transform(df_demo["sequence"].tolist())
dist_matrix.head()


,dist_length,dist_valid_residue_count,dist_charged_count,dist_charged_tercile_bin1_frac,dist_charged_tercile_bin2_frac,dist_charged_tercile_bin3_frac,dist_charged_tercile_cum1,dist_charged_tercile_cum2,dist_charged_tercile_cum3,dist_charged_tercile_entropy,...,dist_pro_quartile_cum2,dist_pro_quartile_cum3,dist_pro_quartile_cum4,dist_pro_quartile_entropy,dist_pro_quartile_max_bin_frac,dist_pro_quartile_spread,dist_pro_quartile_bin1_enrichment,dist_pro_quartile_bin2_enrichment,dist_pro_quartile_bin3_enrichment,dist_pro_quartile_bin4_enrichment
0,24,24,4,0.250000,0.000000,0.750000,0.250000,0.250000,1.0,0.811278,...,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,15,15,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,NaN,...,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,18,18,18,0.277778,0.333333,0.388889,0.277778,0.611111,1.0,1.571542,...,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20,20,5,0.400000,0.400000,0.200000,0.400000,0.800000,1.0,1.521928,...,0.0,1.0,1.0,-0.0,1.0,0.25,-0.25,-0.25,0.75,-0.25
4,19,19,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,NaN,...,1.0,1.0,1.0,-0.0,1.0,0.25,0.75,-0.25,-0.25,-0.25


## Merge transformer output back to the dataset

In [13]:

df_dist_class = pd.concat([df_demo, dist_matrix], axis=1)
df_dist_class.head()


,sequence_id,sequence,label,dist_length,dist_valid_residue_count,dist_charged_count,dist_charged_tercile_bin1_frac,dist_charged_tercile_bin2_frac,dist_charged_tercile_bin3_frac,dist_charged_tercile_cum1,...,dist_pro_quartile_cum2,dist_pro_quartile_cum3,dist_pro_quartile_cum4,dist_pro_quartile_entropy,dist_pro_quartile_max_bin_frac,dist_pro_quartile_spread,dist_pro_quartile_bin1_enrichment,dist_pro_quartile_bin2_enrichment,dist_pro_quartile_bin3_enrichment,dist_pro_quartile_bin4_enrichment
0,dist_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24,24,4,0.250000,0.000000,0.750000,0.250000,...,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,dist_2,GGGGGGGGGGGGGGG,B,15,15,0,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,dist_3,KRRKRRKRRKRRDDDDEE,A,18,18,18,0.277778,0.333333,0.388889,0.277778,...,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,dist_4,ACDEFGHIKLMNPQRSTVWY,B,20,20,5,0.400000,0.400000,0.200000,0.400000,...,0.0,1.0,1.0,-0.0,1.0,0.25,-0.25,-0.25,0.75,-0.25
4,dist_5,PPPPGSSSSSTTTTNNQQQ,A,19,19,0,0.000000,0.000000,0.000000,0.000000,...,1.0,1.0,1.0,-0.0,1.0,0.25,0.75,-0.25,-0.25,-0.25



## Suggested next refactor into the package

A clean migration path into Roxy would be:

- move helper logic into `roxy/sequence/distribution.py`
- keep residue groups in `roxy/core/constants.py`
- expose a class such as `ResidueDistributionDescriptors`
- allow configurable:
  - tracked groups
  - number of bins
  - tercile/quartile/cumulative summaries
- add tests for:
  - empty sequences
  - groups absent from the sequence
  - strongly region-biased sequences
  - lower-case input
  - invalid characters removed during cleaning


## Optional export

In [14]:
# df_dist.to_csv("demo_distribution_descriptors.csv", index=False)
